In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw
from os import listdir,system

In [ ]:
!git clone https://github.com/ultralytics/yolov5

In [ ]:
!pip install -r yolov5/requirements.txt

In [ ]:
!mkdir custom_data
!mkdir custom_data/train_imgs custom_data/test_imgs

In [ ]:
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split

# CSV and image paths
csv_path = "/content/drive/MyDrive/Tunis_detection/labels_plates-annotation_2023-06-05-10-27-56.csv"
image_folder = "/content/drive/MyDrive/Tunis_detection/plates"
destination_folder = "/content/custom_data"

# Output folders
train_img_folder = os.path.join(destination_folder, "train", "images")
train_label_folder = os.path.join(destination_folder, "train", "labels")
val_img_folder = os.path.join(destination_folder, "val", "images")
val_label_folder = os.path.join(destination_folder, "val", "labels")

for folder in [train_img_folder, train_label_folder, val_img_folder, val_label_folder]:
    os.makedirs(folder, exist_ok=True)

# Read CSV annotations
df = pd.read_csv(csv_path)

# Split images (unique image names)
image_names = df["image_name"].unique()
train_imgs, val_imgs = train_test_split(image_names, test_size=0.2, random_state=42)

def convert_to_yolo(x, y, w, h, img_w, img_h):
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    return x_center, y_center, w_norm, h_norm

def process_split(split_imgs, img_dest, label_dest):
    for img_name in split_imgs:
        # Copy image
        src_img_path = os.path.join(image_folder, img_name)
        dst_img_path = os.path.join(img_dest, img_name)
        shutil.copy(src_img_path, dst_img_path)

        # Get rows for this image
        rows = df[df["image_name"] == img_name]
        label_file = os.path.join(label_dest, img_name.replace(".jpg", ".txt"))

        with open(label_file, "w") as f:
            for _, row in rows.iterrows():
                class_id = 0  # Assuming 'tunis' is class 0
                x, y, w, h = row["bbox_x"], row["bbox_y"], row["bbox_width"], row["bbox_height"]
                img_w, img_h = row["image_width"], row["image_height"]
                yolo_box = convert_to_yolo(x, y, w, h, img_w, img_h)
                f.write(f"{class_id} {' '.join(map(str, yolo_box))}\n")

# Process training and validation sets
process_split(train_imgs, train_img_folder, train_label_folder)
process_split(val_imgs, val_img_folder, val_label_folder)

print("✅ Done: Images and YOLO labels split into train/val folders.")


In [ ]:
from PIL import Image, ImageDraw
import os

def getimage(index, Train=True, size=640):
    folderimages = '/content/custom_data/train/images'
    folderlabels = '/content/custom_data/train/labels'

    if not Train:
        folderimages = '/content/custom_data/val/images'
        folderlabels = '/content/custom_data/val/labels'

    images = sorted(os.listdir(folderimages))
    name = images[index][:-4]  # remove .jpg

    # Open image and resize to desired size
    image = Image.open(f'{folderimages}/{name}.jpg').convert('RGB')
    image = image.resize((size, size))  # force to 640x640

    draw = ImageDraw.Draw(image)

    with open(f'{folderlabels}/{name}.txt') as ann:
        for line in ann:
            line = line.strip().split(" ")
            if len(line) < 5:
                continue
            x = float(line[1]) * size
            y = float(line[2]) * size
            w = float(line[3]) * size
            h = float(line[4]) * size
            x0 = int(x - w / 2)
            y0 = int(y - h / 2)
            x1 = int(x + w / 2)
            y1 = int(y + h / 2)
            draw.rectangle([x0, y0, x1, y1], outline="red", width=2)

    return image


In [ ]:
start=10
end=20
fig,axes=plt.subplots(1,end-start,figsize=(60,60))
for i in range(start,end):
    axes[i-start].imshow(np.asarray(getimage(i,True)))

In [ ]:
!cp /content/drive/MyDrive/conf.yaml .

In [ ]:
ann=open('conf.yaml')
line=ann.readline()
list_line=[]
while line !="":
  if('fire'in line):
    line=line.replace('fire','tunis')
  line=line.replace('train_imgs','train')
  line=line.replace('test_imgs','val')
  list_line.append(line)
  line=ann.readline()
file1 = open('conf.yaml', 'w')
file1.writelines(list_line)
file1.close()

In [ ]:
!mv conf.yaml yolov5

In [ ]:
!cd yolov5 && python train.py --img 640 --epochs 15 --data conf.yaml --weights yolov5s.pt

In [ ]:
322!mkdir testing_images && cp custom_data/test_imgs/*.jpg testing_images/

In [ ]:
!cd yolov5 && python detect.py --weights runs/train/exp2/weights/best.pt --source /content/custom_data/val/images

In [ ]:
test_image_files=listdir("yolov5/runs/detect/exp2")

In [ ]:
def get_image_array(name):
    image=Image.open(f'yolov5/runs/detect/exp2/{name}')
    return np.asarray(image)

In [ ]:
start=30
end=40
fig,axes=plt.subplots(1,end-start,figsize=(60,60))
for i in range(start,end):
    axes[i-start].imshow(get_image_array(test_image_files[i]))

In [ ]:
!zip -r resultv2.zip yolov5

In [ ]:
!mv resultv2.zip /content/drive/MyDrive/Tunis_detection

In [ ]:
import zipfile

model_zip_path = "/content/drive/MyDrive/Tunis_detection/resultv2.zip"
extract_dir = "/content/resultv2"

with zipfile.ZipFile(model_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)


In [ ]:
import torch
from matplotlib import pyplot as plt
import cv2

# Clone YOLOv5 (needed for model architecture only)
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt


In [ ]:
import torch
word_Tunisia="تونس"
!pip install easyocr
from matplotlib import pyplot as plt
import cv2

In [ ]:

# Load the model using torch hub (no CLI needed)
model_path = "/content/resultv2/yolov5/runs/train/exp2/weights/best.pt"  # adjust if needed
model = torch.hub.load("ultralytics/yolov5", "custom", path=model_path, force_reload=True)




In [ ]:
# Run inference on an image
# Ensure Matplotlib inline rendering is active
%matplotlib inline

from PIL import Image
import matplotlib.pyplot as plt
import os

img_path = "/content/drive/MyDrive/Tunis_detection/plates/148.jpg"

# Check and display
if os.path.exists(img_path):
    image = Image.open(img_path)
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title("Original Image")
    plt.show()
else:
    print("❌ Image not found.")

results = model(img_path)
import easyocr

reader = easyocr.Reader(['en'])
boxes = results.xyxy[0]  # tensor: (x1, y1, x2, y2, conf, class)
image = cv2.imread(img_path)
if len(boxes) > 0:
    # Assume only one detection (first one)
    x1, y1, x2, y2, conf, cls = boxes[0].tolist()
    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

    # Left region (from start to x1)
    left_crop = image[3:-3, 2:x1-4]

    # Right region (from x2 to end)
    right_crop = image[3:-3, x2+2:-14]
    left_result = reader.readtext(left_crop, detail=0)
    right_result = reader.readtext(right_crop, detail=0)
    left_text = " ".join(left_result)
    right_text = " ".join(right_result)
    # Combine in RTL order visually
    text = f"{right_text} {word_Tunisia} {left_text}"
    print("✅ Final OCR Output:", text)

    print("✅ Object detected.")
else:
    print("❌ No object detected.")

# Get detection details
for *box, conf, cls in results.xyxy[0]:  # xyxy format
    print(f"Box: {box}, Confidence: {conf:.2f}, Class: {int(cls)}")
